In [1]:
!pip install pandas numpy scikit-learn scipy

In [2]:
import pandas as pd
import os
import numpy as np
from sklearn.preprocessing import StandardScaler 
from scipy.stats.mstats import winsorize
from pathlib import Path
import sys

pasta_principal = str(Path().resolve().parent)
if pasta_principal not in sys.path:
    sys.path.append(pasta_principal)

from src.data_loader import carregar_dados_brutos
from src.preprocessing import (
    corrigir_valores_invalidos,
    filtrar_colunas,
    transformar_categoricas_em_numeros,
)

In [3]:
dataLimpo = carregar_dados_brutos('../data/raw/college_sleep_and_gpa.csv')

In [4]:
invalidos = dataLimpo[~dataLimpo['first_generation'].isin([0, 1]) & dataLimpo['first_generation'].notna()]['first_generation']

if not invalidos.empty:
    for val in invalidos.unique():
        print(f"Valor inválido encontrado na coluna 'first_generation': {val}")
    
    dataLimpo = corrigir_valores_invalidos(dataLimpo)
    print("O valor 2 foi substituído por 1 com sucesso.")

Valor inválido encontrado na coluna 'first_generation': 2.0
O valor 2 foi substituído por 1 com sucesso.


In [5]:
colInutil = [
    'avg_sleep_minutes',
    'term_units',
    'student_id',
    'sleep_midpoint_clock',
    'cohort_code',
    'study',
    'semester',
]
colunas_mantidas = [coluna for coluna in dataLimpo.columns if coluna not in colInutil]
dataLimpo = filtrar_colunas(dataLimpo, colunas_mantidas)

In [6]:
dataLimpo = dataLimpo.dropna(subset=['gender', 'first_generation', 'underrepresented'])

In [7]:
dataLimpo['term_load_z'] = dataLimpo['term_load_z'].fillna(dataLimpo['term_load_z'].median())
print(dataLimpo.isnull().sum())

university                 0
gender                     0
first_generation           0
underrepresented           0
avg_sleep_hours            0
daytime_sleep_minutes      0
sleep_midpoint_minutes     0
bedtime_variability        0
nights_tracked_fraction    0
prior_gpa                  0
term_gpa                   0
gpa_change                 0
term_load_z                0
under_6h_sleep             0
sleep_bracket              0
dtype: int64


In [8]:
transfLog =['bedtime_variability', 'daytime_sleep_minutes']
col_win_inf = ['prior_gpa', 'term_gpa']

for col in transfLog:
    dataLimpo[col] = np.log1p(dataLimpo[col])

dataLimpo['sleep_midpoint_minutes'] = winsorize(dataLimpo['sleep_midpoint_minutes'], limits=[0, 0.03])

for col in col_win_inf:
    dataLimpo[col] = winsorize(dataLimpo[col], limits=[0.05, 0])

dataLimpo['term_load_z'] = winsorize(dataLimpo['term_load_z'], limits=[0.03, 0.03])


In [9]:
mediana_gpa = dataLimpo['term_gpa'].median()
dataLimpo['gpa_category'] = np.where(dataLimpo['term_gpa'] >= mediana_gpa, 'Alto', 'Baixo')

print(dataLimpo['gpa_category'].value_counts())

gpa_category
Alto     318
Baixo    308
Name: count, dtype: int64


/home/henderson/Documentos/AnaliseDeDadosSono/AnaliseDeDados/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:840: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)


In [10]:
dataEncod = dataLimpo.copy()

In [11]:
bracketsEncod = {
    '<5.5h': 1,
    '5.5-6h': 2,
    '6-6.5h': 3,
    '6.5-7h': 4,
    '7-7.5h': 5,
    '7.5h+': 6,
}

dataEncod['sleep_bracket_encoded'] = dataEncod['sleep_bracket'].map(bracketsEncod)
dataEncod = dataEncod.drop(columns=['sleep_bracket'])


In [12]:
dataEncod['gpa_category_encoded'] = dataEncod['gpa_category'].map({'Alto': 1, 'Baixo': 0})


In [13]:

dataEncod = transformar_categoricas_em_numeros(
    dataEncod, colunas=['gender', 'university']
)

dataEncod.head()

,first_generation,underrepresented,avg_sleep_hours,daytime_sleep_minutes,sleep_midpoint_minutes,bedtime_variability,nights_tracked_fraction,prior_gpa,term_gpa,gpa_change,term_load_z,under_6h_sleep,gpa_category,sleep_bracket_encoded,gpa_category_encoded,gender_Male,university_University of Notre Dame,university_University of Washington
0,0.0,0.0,6.17,2.541602,500.3,0.048790,0.931,4.00,3.76,-0.24,0.112,0,Alto,3,1,0,0,0
1,0.0,0.0,6.78,3.925926,359.5,0.046502,0.862,4.00,3.55,-0.45,0.112,0,Baixo,4,0,0,0,0
2,0.0,1.0,4.23,5.190175,398.2,0.151776,0.586,3.00,3.59,0.59,-0.604,1,Alto,1,1,0,0,0
3,0.0,0.0,6.86,3.299534,532.1,0.579642,0.931,3.80,3.46,-0.34,-1.805,0,Baixo,4,0,0,0,0
4,0.0,1.0,6.80,4.284965,354.0,0.143061,0.931,3.17,3.50,0.33,-1.805,0,Baixo,4,0,0,0,0


In [14]:
scaler = StandardScaler()

colunas_continuas = [
    'avg_sleep_hours',
    'daytime_sleep_minutes',
    'sleep_midpoint_minutes',
    'bedtime_variability',
    'nights_tracked_fraction',
    'prior_gpa',
    'gpa_change',
    'term_load_z'
]

dataEncod[colunas_continuas] = scaler.fit_transform(dataEncod[colunas_continuas])

dataEncod.describe()

/home/henderson/Documentos/AnaliseDeDadosSono/AnaliseDeDados/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:4798: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
/home/henderson/Documentos/AnaliseDeDadosSono/AnaliseDeDados/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:4798: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
/home/henderson/Documentos/AnaliseDeDadosSono/AnaliseDeDados/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:4798: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(


,first_generation,underrepresented,avg_sleep_hours,daytime_sleep_minutes,sleep_midpoint_minutes,bedtime_variability,nights_tracked_fraction,prior_gpa,term_gpa,gpa_change,term_load_z,under_6h_sleep,sleep_bracket_encoded,gpa_category_encoded,gender_Male,university_University of Notre Dame,university_University of Washington
count,626.000000,626.000000,6.260000e+02,6.260000e+02,6.260000e+02,6.260000e+02,6.260000e+02,6.260000e+02,626.000000,6.260000e+02,6.260000e+02,626.000000,626.000000,626.000000,626.000000,626.000000,626.000000
mean,0.166134,0.188498,-5.221241e-16,-2.724126e-16,5.675261e-18,4.540209e-17,-3.859178e-16,-6.810314e-16,3.474272,8.512892e-18,9.931708e-18,0.207668,3.758786,0.507987,0.418530,0.231629,0.439297
std,0.372499,0.391422,1.000800e+00,1.000800e+00,1.000800e+00,1.000800e+00,1.000800e+00,1.000800e+00,0.424842,1.000800e+00,1.000800e+00,0.405962,1.451386,0.500336,0.493713,0.422211,0.496698
min,0.000000,0.000000,-3.989062e+00,-3.944992e+00,-2.196071e+00,-6.878197e-01,-3.649348e+00,-2.246439e+00,2.490000,-4.600616e+00,-2.251545e+00,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,-6.036795e-01,-6.384084e-01,-7.590404e-01,-5.064263e-01,-2.596457e-01,-6.516382e-01,3.233000,-4.292172e-01,-6.097322e-01,0.000000,3.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,6.748378e-02,2.921435e-02,-1.462053e-01,-3.549026e-01,3.658019e-01,1.979285e-01,3.556000,3.921539e-02,3.584098e-02,0.000000,4.000000,1.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,6.469904e-01,6.548200e-01,5.955706e-01,-7.697589e-04,7.399536e-01,7.941156e-01,3.810000,5.181538e-01,4.472235e-01,0.000000,5.000000,1.000000,1.000000,0.000000,1.000000
max,1.000000,1.000000,3.745577e+00,3.513592e+00,2.401841e+00,7.706269e+00,7.399536e-01,1.338460e+00,4.000000,3.331839e+00,2.066732e+00,1.000000,6.000000,1.000000,1.000000,1.000000,1.000000


In [15]:
caminho_dados_processados = '../data/processed/'
os.makedirs(caminho_dados_processados, exist_ok=True)
dataEncod.to_csv(f'{caminho_dados_processados}dataset_clean.csv', index=False)